## 1. Install dependencies

This cell installs the libraries required for our miniature RAG implementation. We will use Hugging Face Transformers and Datasets for NLP processing, FAISS for vector search in later levels, and PyTorch as the deep-learning framework.

In [1]:
!pip -q install transformers datasets sentence-transformers faiss-cpu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 29.9 MB/s eta 0:00:00


## 2. Configure the environment

This cell imports the libraries used throughout the project and selects the available hardware. A GPU will be used automatically if Colab provides one; otherwise, the implementation will run on the CPU.

In [2]:
import random
import numpy as np
import torch

from datasets import Dataset, DatasetDict

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.11.0+cpu
Device: cpu


## 3. Create the knowledge base

RAG retrieves information from an external collection of documents. Here, we create a small knowledge base containing factual passages from different topics. Each passage has a unique identifier and will act as a retrievable document.

The collection is intentionally much smaller than the Wikipedia index used in the paper, making it practical for experimentation in Google Colab.

In [3]:
documents = [
    {
        "id": "doc_001",
        "title": "Photosynthesis",
        "text": (
            "Photosynthesis is the process by which green plants, algae, "
            "and some bacteria convert light energy into chemical energy. "
            "Plants generally use carbon dioxide and water to produce "
            "glucose and oxygen."
        ),
    },
    {
        "id": "doc_002",
        "title": "Mitochondria",
        "text": (
            "Mitochondria are organelles found in most eukaryotic cells. "
            "They generate much of the cell's ATP through cellular "
            "respiration and are often called the powerhouses of the cell."
        ),
    },
    {
        "id": "doc_003",
        "title": "Water Cycle",
        "text": (
            "The water cycle describes the continuous movement of water "
            "between Earth's surface and the atmosphere. Major processes "
            "include evaporation, condensation, precipitation, and collection."
        ),
    },
    {
        "id": "doc_004",
        "title": "Solar System",
        "text": (
            "The Solar System consists of the Sun and the objects that "
            "orbit it, including eight planets, dwarf planets, moons, "
            "asteroids, and comets. The Sun is its central star."
        ),
    },
    {
        "id": "doc_005",
        "title": "Machine Learning",
        "text": (
            "Machine learning is a branch of artificial intelligence in "
            "which models learn patterns from data. Supervised learning "
            "uses examples with known target outputs."
        ),
    },
    {
        "id": "doc_006",
        "title": "Neural Networks",
        "text": (
            "A neural network is composed of interconnected computational "
            "units called neurons. During training, its parameters are "
            "adjusted to reduce a specified loss function."
        ),
    },
    {
        "id": "doc_007",
        "title": "World Wide Web",
        "text": (
            "The World Wide Web is a system of interconnected documents "
            "and resources accessed through the internet. Tim Berners-Lee "
            "proposed the Web in 1989."
        ),
    },
    {
        "id": "doc_008",
        "title": "Roman Empire",
        "text": (
            "The Roman Empire was an ancient state centered on the city "
            "of Rome. Its territories extended across parts of Europe, "
            "North Africa, and western Asia."
        ),
    },
    {
        "id": "doc_009",
        "title": "Plate Tectonics",
        "text": (
            "Plate tectonics is the scientific theory that Earth's "
            "lithosphere is divided into moving plates. Their movement "
            "can produce earthquakes, mountains, and volcanoes."
        ),
    },
    {
        "id": "doc_010",
        "title": "DNA",
        "text": (
            "Deoxyribonucleic acid, or DNA, stores hereditary genetic "
            "information in living organisms. Its structure consists "
            "of two complementary strands forming a double helix."
        ),
    },
]

print("Number of documents:", len(documents))
print(documents[0])

Number of documents: 10
{'id': 'doc_001', 'title': 'Photosynthesis', 'text': 'Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy into chemical energy. Plants generally use carbon dioxide and water to produce glucose and oxygen.'}


## 4. Inspect the documents

This cell checks that each document has a unique identifier, a title, and textual content. Inspecting the raw data helps us detect missing fields or formatting problems before creating the retrieval index.

In [4]:
for document in documents:
    assert document["id"]
    assert document["title"]
    assert document["text"]

print("All documents passed validation.")
print("Average passage length:",
      np.mean([len(d["text"].split()) for d in documents]))

for document in documents[:3]:
    print(f"\n{document['id']} — {document['title']}")
    print(document["text"])

All documents passed validation.
Average passage length: 24.6

doc_001 — Photosynthesis
Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy into chemical energy. Plants generally use carbon dioxide and water to produce glucose and oxygen.

doc_002 — Mitochondria
Mitochondria are organelles found in most eukaryotic cells. They generate much of the cell's ATP through cellular respiration and are often called the powerhouses of the cell.

doc_003 — Water Cycle
The water cycle describes the continuous movement of water between Earth's surface and the atmosphere. Major processes include evaporation, condensation, precipitation, and collection.


## 5. Load the embedding model

This cell loads a pretrained sentence-transformer that converts text into fixed-length dense vectors. The same embedding space will later be used for both questions and documents.

In [5]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


## 6. Prepare document text

This cell combines each document's title and passage into the text that will be embedded. Including the title provides additional information that can help distinguish documents with similar content.

In [6]:
document_texts = [
    f"{document['title']}. {document['text']}"
    for document in documents
]

print(document_texts[0])

Photosynthesis. Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy into chemical energy. Plants generally use carbon dioxide and water to produce glucose and oxygen.


## 7. Generate document embeddings

This cell encodes every knowledge-base document into a dense numerical vector. These vectors form the representation of our non-parametric memory and will later be stored in a vector index.

In [7]:
document_embeddings = embedding_model.encode(
    document_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", document_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (10, 384)


## 8. Inspect the embedding space

This cell verifies the dimensions and numerical structure of the generated embeddings. Each document should have one vector with the same dimensionality.

In [8]:
print("Number of document embeddings:", len(document_embeddings))
print("Embedding dimension:", document_embeddings.shape[1])
print("First embedding:")
print(document_embeddings[0][:10])

Number of document embeddings: 10
Embedding dimension: 384
First embedding:
[-0.05568096  0.08985692 -0.07161225  0.05245663  0.01987322  0.00848286
  0.018645   -0.02772993  0.05177807  0.09027112]


## 9. Build the FAISS index

This cell creates a FAISS vector index containing the document embeddings. Because the embeddings are normalized, inner-product similarity corresponds to cosine similarity. This provides the vector-search mechanism used to retrieve the most relevant passages.

The original RAG system uses a much larger FAISS MIPS index containing approximately 21 million Wikipedia passages. Our index is intentionally much smaller for Colab.

In [9]:
import faiss

embedding_dim = document_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(document_embeddings.astype("float32"))

print("Documents indexed:", index.ntotal)

Documents indexed: 10


## 10. Test the retrieval index

This cell performs a simple semantic search using a sample question. The question is converted into the same embedding space as the documents, and FAISS returns the documents with the highest inner-product similarity.

In [10]:
query = "What process allows green plants to convert light energy into chemical energy?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = index.search(query_embedding, k=3)

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    document = documents[idx]

    print(f"\nRank {rank}")
    print("Score:", round(float(score), 4))
    print("Document:", document["title"])
    print("Text:", document["text"])


Rank 1
Score: 0.7885
Document: Photosynthesis
Text: Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy into chemical energy. Plants generally use carbon dioxide and water to produce glucose and oxygen.

Rank 2
Score: 0.2345
Document: Water Cycle
Text: The water cycle describes the continuous movement of water between Earth's surface and the atmosphere. Major processes include evaporation, condensation, precipitation, and collection.

Rank 3
Score: 0.1756
Document: Mitochondria
Text: Mitochondria are organelles found in most eukaryotic cells. They generate much of the cell's ATP through cellular respiration and are often called the powerhouses of the cell.


## 11. Create the retrieval function

This cell defines a reusable dense-retrieval function. A question is converted into the same embedding space as the documents, then FAISS returns the Top-K most similar passages.

In [11]:
def retrieve_documents(query, k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "id": documents[idx]["id"],
            "title": documents[idx]["title"],
            "text": documents[idx]["text"],
            "score": float(score)
        })

    return results

## 12. Test retrieval on multiple questions

This cell tests the retriever on several questions covering different documents. The goal is to verify that semantically relevant passages are ranked near the top.

In [12]:
test_queries = [
    "How do plants produce food using sunlight?",
    "What is the powerhouse of the cell?",
    "How does water move through evaporation and precipitation?",
    "What is machine learning?",
    "What does DNA contain?"
]

for query in test_queries:
    print("\n" + "=" * 70)
    print("QUERY:", query)

    results = retrieve_documents(query, k=3)

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank}. {result['title']} "
            f"(score={result['score']:.4f})"
        )


QUERY: How do plants produce food using sunlight?
1. Photosynthesis (score=0.5974)
2. Water Cycle (score=0.2104)
3. Mitochondria (score=0.1989)

QUERY: What is the powerhouse of the cell?
1. Mitochondria (score=0.5464)
2. Photosynthesis (score=0.3059)
3. DNA (score=0.1996)

QUERY: How does water move through evaporation and precipitation?
1. Water Cycle (score=0.6121)
2. Photosynthesis (score=0.1901)
3. Plate Tectonics (score=0.1355)

QUERY: What is machine learning?
1. Machine Learning (score=0.8422)
2. Neural Networks (score=0.4019)
3. World Wide Web (score=0.2287)

QUERY: What does DNA contain?
1. DNA (score=0.6780)
2. Mitochondria (score=0.2298)
3. Photosynthesis (score=0.1600)


## 13. Inspect retrieved passages

This cell displays the actual retrieved passages rather than only their titles and scores. This lets us verify whether the retriever is finding useful evidence for a question.

In [13]:
query = "How do plants produce food using sunlight?"

results = retrieve_documents(query, k=3)

for rank, result in enumerate(results, start=1):
    print(f"\n--- Rank {rank} ---")
    print("Title:", result["title"])
    print("Score:", round(result["score"], 4))
    print("Passage:", result["text"])


--- Rank 1 ---
Title: Photosynthesis
Score: 0.5974
Passage: Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy into chemical energy. Plants generally use carbon dioxide and water to produce glucose and oxygen.

--- Rank 2 ---
Title: Water Cycle
Score: 0.2104
Passage: The water cycle describes the continuous movement of water between Earth's surface and the atmosphere. Major processes include evaporation, condensation, precipitation, and collection.

--- Rank 3 ---
Title: Mitochondria
Score: 0.1989
Passage: Mitochondria are organelles found in most eukaryotic cells. They generate much of the cell's ATP through cellular respiration and are often called the powerhouses of the cell.


## 14. Check Top-K retrieval accuracy

This cell measures whether the expected document appears within the Top-K retrieved results. This is a simple retrieval metric that we will use before connecting retrieval to the generator.

In [14]:
evaluation_queries = [
    ("What process allows plants to convert sunlight into energy?",
     "doc_001"),

    ("Which organelle produces most of the cell's ATP?",
     "doc_002"),

    ("What causes rain to fall from clouds?",
     "doc_003"),

    ("What is machine learning used for?",
     "doc_005"),

    ("What molecule carries genetic information?",
     "doc_010")
]

def retrieval_hit_rate(queries, k):
    hits = 0

    for query, expected_id in queries:
        results = retrieve_documents(query, k=k)
        retrieved_ids = [result["id"] for result in results]

        if expected_id in retrieved_ids:
            hits += 1

    return hits / len(queries)


for k in [1, 3, 5]:
    hit_rate = retrieval_hit_rate(evaluation_queries, k)
    print(f"Top-{k} Hit Rate: {hit_rate:.2%}")

Top-1 Hit Rate: 100.00%
Top-3 Hit Rate: 100.00%
Top-5 Hit Rate: 100.00%


## 15. Inspect the Top-K result for one query

This final cell gives us a compact view of the retriever's ranked output. At this point, retrieval is independent of the generator and can provide the evidence passages that RAG will condition on.

In [15]:
query = "What molecule stores hereditary information?"

results = retrieve_documents(query, k=5)

for rank, result in enumerate(results, start=1):
    print(
        f"{rank}. {result['title']} "
        f"| similarity={result['score']:.4f}"
    )

1. DNA | similarity=0.6464
2. Mitochondria | similarity=0.2865
3. Photosynthesis | similarity=0.1428
4. Machine Learning | similarity=0.1408
5. Neural Networks | similarity=0.1120


## 16. Load the generator tokenizer

This cell loads the tokenizer for a pretrained sequence-to-sequence model. The tokenizer converts the question and retrieved passage into token IDs that the generator can process.

In [16]:
from transformers import AutoTokenizer

GENERATOR_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_NAME)

print("Tokenizer loaded.")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Tokenizer loaded.


## 17. Create the RAG input format

This cell combines a question with one retrieved passage. This follows the core RAG idea of conditioning the generator on both the input and retrieved evidence.

In [17]:
def create_generator_input(query, retrieved_document):
    return (
        f"question: {query} "
        f"context: {retrieved_document['text']}"
    )

## 18. Retrieve and format Top-K passages

This cell retrieves the Top-K passages for a question and converts each question-passage pair into a generator input. Each retrieved document is therefore treated as a separate candidate source of evidence.

In [18]:
query = "What process allows plants to convert sunlight into energy?"

retrieved_documents = retrieve_documents(query, k=3)

generator_inputs = [
    create_generator_input(query, document)
    for document in retrieved_documents
]

for rank, text in enumerate(generator_inputs, start=1):
    print(f"\n--- Retrieved document {rank} ---")
    print(text)


--- Retrieved document 1 ---
question: What process allows plants to convert sunlight into energy? context: Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy into chemical energy. Plants generally use carbon dioxide and water to produce glucose and oxygen.

--- Retrieved document 2 ---
question: What process allows plants to convert sunlight into energy? context: The water cycle describes the continuous movement of water between Earth's surface and the atmosphere. Major processes include evaporation, condensation, precipitation, and collection.

--- Retrieved document 3 ---
question: What process allows plants to convert sunlight into energy? context: Mitochondria are organelles found in most eukaryotic cells. They generate much of the cell's ATP through cellular respiration and are often called the powerhouses of the cell.


## 19. Tokenize the generator inputs

This cell tokenizes the question-passage pairs and converts them into tensors suitable for the seq2seq generator.

In [19]:
inputs = tokenizer(
    generator_inputs,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

print("Input tensor shape:", inputs["input_ids"].shape)

Input tensor shape: torch.Size([3, 62])


## 20. Verify the final generator input

This cell decodes one tokenized input back into text so we can verify that the generator receives the expected question and retrieved evidence.

In [20]:
decoded_input = tokenizer.decode(
    inputs["input_ids"][0],
    skip_special_tokens=True
)

print(decoded_input)

question: What process allows plants to convert sunlight into energy? context: Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy into chemical energy. Plants generally use carbon dioxide and water to produce glucose and oxygen.


## 21. Load the generator

This cell loads the pretrained seq2seq generator. At this stage, the generator is frozen; we are testing retrieval-augmented generation before implementing RAG training.

In [21]:
from transformers import AutoModelForSeq2SeqLM

generator = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATOR_NAME
).to(device)

generator.eval()

print("Generator loaded.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generator loaded.


## 22. Generate an answer from one retrieved passage

This cell gives the generator one question together with the highest-ranked retrieved passage and generates an answer from that evidence.

In [22]:
query = "What process allows plants to convert sunlight into energy?"

retrieved_document = retrieve_documents(query, k=1)[0]

generator_input = create_generator_input(
    query,
    retrieved_document
)

inputs = tokenizer(
    generator_input,
    return_tensors="pt",
    truncation=True,
    max_length=256
).to(device)

with torch.no_grad():
    output_ids = generator.generate(
        **inputs,
        max_new_tokens=64
    )

answer = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("Question:", query)
print("Retrieved:", retrieved_document["title"])
print("Answer:", answer)

Question: What process allows plants to convert sunlight into energy?
Retrieved: Photosynthesis
Answer: Photosynthesis


## 23. Generate using multiple retrieved passages

This cell generates an answer separately for each of the Top-K retrieved passages. These candidate answers will later be combined using the RAG marginalization idea.

In [23]:
query = "What process allows plants to convert sunlight into energy?"

retrieved_documents = retrieve_documents(query, k=3)

for rank, document in enumerate(retrieved_documents, start=1):

    generator_input = create_generator_input(
        query,
        document
    )

    inputs = tokenizer(
        generator_input,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        output_ids = generator.generate(
            **inputs,
            max_new_tokens=64
        )

    answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    print(f"\n--- Document {rank}: {document['title']} ---")
    print("Similarity:", round(document["score"], 4))
    print("Answer:", answer)


--- Document 1: Photosynthesis ---
Similarity: 0.7432
Answer: Photosynthesis

--- Document 2: Water Cycle ---
Similarity: 0.3106
Answer: condensation

--- Document 3: Mitochondria ---
Similarity: 0.2096
Answer: cellular respiration


## 24. Create a reusable RAG generation function

This cell packages retrieval and generation into one function. Given a question, it retrieves Top-K passages and generates an answer conditioned on each passage.

In [24]:
def rag_generate(query, k=3):

    retrieved_documents = retrieve_documents(query, k=k)

    results = []

    for document in retrieved_documents:

        generator_input = create_generator_input(
            query,
            document
        )

        inputs = tokenizer(
            generator_input,
            return_tensors="pt",
            truncation=True,
            max_length=256
        ).to(device)

        with torch.no_grad():
            output_ids = generator.generate(
                **inputs,
                max_new_tokens=64
            )

        answer = tokenizer.decode(
            output_ids[0],
            skip_special_tokens=True
        )

        results.append({
            "document": document,
            "answer": answer
        })

    return results

## 25. Test the complete pipeline

This cell tests the complete miniature RAG pipeline: question → dense retrieval → Top-K passages → generator → candidate answers.

In [25]:
query = "What molecule stores hereditary information?"

results = rag_generate(query, k=3)

print("QUESTION:", query)

for rank, result in enumerate(results, start=1):
    document = result["document"]

    print(f"\n--- Rank {rank} ---")
    print("Retrieved:", document["title"])
    print("Similarity:", round(document["score"], 4))
    print("Answer:", result["answer"])

QUESTION: What molecule stores hereditary information?

--- Rank 1 ---
Retrieved: DNA
Similarity: 0.6464
Answer: Deoxyribonucleic acid

--- Rank 2 ---
Retrieved: Mitochondria
Similarity: 0.2865
Answer: mitochondria

--- Rank 3 ---
Retrieved: Photosynthesis
Similarity: 0.1428
Answer: chromosome


## 27. Load the larger SQuAD dataset

This cell downloads SQuAD 1.1 and converts the official training examples into a Hugging Face Dataset. Each example contains a question, its source passage, and the answer span.

In [38]:
import requests
from datasets import Dataset

SQUAD_URL = (
    "https://rajpurkar.github.io/"
    "SQuAD-explorer/dataset/train-v1.1.json"
)

response = requests.get(SQUAD_URL)
response.raise_for_status()

squad_json = response.json()

train_records = []

for article in squad_json["data"]:
    title = article["title"]

    for paragraph in article["paragraphs"]:
        context = paragraph["context"]

        for qa in paragraph["qas"]:
            train_records.append({
                "id": qa["id"],
                "title": title,
                "context": context,
                "question": qa["question"],
                "answers": qa["answers"]
            })

squad_train = Dataset.from_list(train_records)

print(squad_train)
print("Total examples:", len(squad_train))

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 87599
})
Total examples: 87599


## 28. Create a Colab-sized training and validation set

The full SQuAD training set is much larger than necessary for our educational reproduction. This cell selects 5,000 training examples and 500 validation examples while keeping the dataset large enough to move beyond the earlier 10-example toy setup.

In [39]:
SEED = 42

shuffled = squad_train.shuffle(seed=SEED)

train_size = 5000
val_size = 500

rag_train = shuffled.select(range(train_size))
rag_val = shuffled.select(
    range(train_size, train_size + val_size)
)

print("Training examples:", len(rag_train))
print("Validation examples:", len(rag_val))

Training examples: 5000
Validation examples: 500


## 29. Build the retrieval corpus

The retriever needs a non-parametric knowledge base. Here we collect the unique SQuAD contexts appearing in our selected examples and use them as the document corpus.

In [40]:
unique_contexts = {}
context_id = 0

for example in rag_train:
    context = example["context"]

    if context not in unique_contexts:
        unique_contexts[context] = {
            "id": f"squad_{context_id}",
            "title": example["title"],
            "text": context
        }
        context_id += 1

rag_documents = list(unique_contexts.values())

print("Unique retrieval documents:", len(rag_documents))
print("Example document:")
print(rag_documents[0]["text"][:500])

Unique retrieval documents: 4426
Example document:
The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supporte


## 30. Build the larger FAISS retrieval index

This cell embeds the larger retrieval corpus and rebuilds the FAISS index. Unlike the earlier 10-document experiment, the final RAG model now retrieves from thousands of real Wikipedia-derived passages.

In [41]:
rag_document_texts = [
    f"{document['title']}. {document['text']}"
    for document in rag_documents
]

rag_document_embeddings = embedding_model.encode(
    rag_document_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64
).astype("float32")

rag_embedding_dim = rag_document_embeddings.shape[1]

rag_index = faiss.IndexFlatIP(rag_embedding_dim)
rag_index.add(rag_document_embeddings)

print("Indexed documents:", rag_index.ntotal)
print("Embedding dimension:", rag_embedding_dim)

Batches:   0%|          | 0/70 [00:00<?, ?it/s]

Indexed documents: 4426
Embedding dimension: 384


## 31. Create the training retrieval function

This cell retrieves the Top-K documents for a training question using the larger FAISS index. These retrieved documents will form the latent document set used by the RAG loss.

In [43]:
def rag_retrieve(query, k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = rag_index.search(
        query_embedding,
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "id": rag_documents[idx]["id"],
            "title": rag_documents[idx]["title"],
            "text": rag_documents[idx]["text"],
            "score": float(score)
        })

    return results

## 32. Define the RAG-Sequence marginal likelihood

This is the central RAG training step. For each question, the model retrieves Top-K documents, computes the generator probability of the correct answer given each document, and marginalizes over the retrieved documents. This corresponds to the RAG-Sequence objective where one document is used for the complete generated sequence.

In [44]:
import torch
import torch.nn.functional as F

def rag_sequence_loss(
    query,
    answer,
    k=3
):
    retrieved = rag_retrieve(query, k=k)

    generator_inputs = [
        create_generator_input(query, document)
        for document in retrieved
    ]

    encoded_inputs = tokenizer(
        generator_inputs,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    ).to(device)

    labels = tokenizer(
        [answer] * len(retrieved),
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    ).input_ids.to(device)

    labels[labels == tokenizer.pad_token_id] = -100

    outputs = generator(
        **encoded_inputs,
        labels=labels
    )

    logits = outputs.logits

    token_log_probs = F.log_softmax(
        logits,
        dim=-1
    )

    safe_labels = labels.clone()
    safe_labels[safe_labels == -100] = 0

    selected_log_probs = torch.gather(
        token_log_probs,
        2,
        safe_labels.unsqueeze(-1)
    ).squeeze(-1)

    mask = labels != -100

    sequence_log_probs = (
        selected_log_probs * mask
    ).sum(dim=1)

    # Retriever probability p_eta(z|x)
    retrieval_scores = torch.tensor(
        [doc["score"] for doc in retrieved],
        device=device,
        dtype=sequence_log_probs.dtype
    )

    retrieval_log_probs = F.log_softmax(
        retrieval_scores,
        dim=0
    )

    # RAG-Sequence marginal likelihood:
    # log sum_z p(z|x) p(y|x,z)
    joint_log_probs = (
        retrieval_log_probs +
        sequence_log_probs
    )

    marginal_log_likelihood = torch.logsumexp(
        joint_log_probs,
        dim=0
    )

    return -marginal_log_likelihood

## 33. Test the RAG loss

Before starting training, this cell runs one real SQuAD example through retrieval, generation, and the RAG marginal likelihood calculation.

In [46]:
example = rag_train[0]

question = example["question"]
answer = example["answers"][0]["text"][0]

loss = rag_sequence_loss(
    question,
    answer,
    k=3
)

print("Question:", question)
print("Answer:", answer)
print("Initial RAG loss:", loss.item())

Question: What percentage of Egyptians polled support death penalty for those leaving Islam?
Answer: 8
Initial RAG loss: 7.498862266540527


## 34. Prepare the optimizer

The paper keeps the document encoder and document index fixed while fine-tuning the query-side retrieval component and generator. In our miniature implementation, the sentence-transformer retriever is frozen, so we train the FLAN-T5 generator.

In [47]:
generator.train()

optimizer = torch.optim.AdamW(
    generator.parameters(),
    lr=2e-5
)

print("Trainable parameters prepared.")

Trainable parameters prepared.


## 35. Train the miniature RAG model

This cell trains the generator using the RAG-Sequence loss. We use one epoch initially because every example performs multiple generator forward passes for its retrieved documents. After confirming that training works, the number of epochs can be increased.

In [ ]:
from tqdm.auto import tqdm

EPOCHS = 1
K = 3

for epoch in range(EPOCHS):

    total_loss = 0.0

    progress = tqdm(
        rag_train,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for example in progress:

        question = example["question"]
        answer = example["answers"][0]["text"][0]

        optimizer.zero_grad()

        loss = rag_sequence_loss(
            question,
            answer,
            k=K
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_loss = total_loss / len(rag_train)

    print(
        f"\nEpoch {epoch + 1} "
        f"Average Loss: {average_loss:.4f}"
    )

Epoch 1/1:   0%|          | 0/5000 [00:00<?, ?it/s]

## 36. Save the trained generator

This cell saves the trained generator and tokenizer so the final RAG model can be evaluated without retraining.

In [ ]:
SAVE_PATH = "./mini_rag_generator"

generator.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("RAG generator saved to:", SAVE_PATH)

## 37. Put the trained generator into evaluation mode

This cell switches the trained generator to evaluation mode. The retrieval index remains available as the model's external non-parametric memory.

In [ ]:
generator.eval()

print("RAG model ready for evaluation.")

## 38. Prepare the validation set

This cell extracts questions and reference answers from the held-out validation set. These examples were not used during training.

In [ ]:
evaluation_examples = rag_val

print("Evaluation examples:", len(evaluation_examples))
print(evaluation_examples[0]["question"])
print(evaluation_examples[0]["answers"]["text"][0])

## 39. Generate answers with RAG

This cell retrieves the Top-K passages for each validation question and generates an answer using the trained RAG generator.

In [ ]:
def generate_rag_answer(query, k=3):

    retrieved = rag_retrieve(query, k=k)

    inputs = [
        create_generator_input(query, document)
        for document in retrieved
    ]

    encoded = tokenizer(
        inputs,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = generator.generate(
            **encoded,
            max_new_tokens=64
        )

    answers = [
        tokenizer.decode(
            output,
            skip_special_tokens=True
        )
        for output in outputs
    ]

    return answers[0], retrieved

## 40. Evaluate Exact Match

This cell calculates Exact Match (EM), which checks whether the predicted answer exactly matches the normalized reference answer.

In [ ]:
import re

def normalize_answer(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = " ".join(text.split())
    return text


def exact_match(prediction, references):

    prediction = normalize_answer(prediction)

    return any(
        prediction == normalize_answer(reference)
        for reference in references
    )

## 41. Evaluate token-level F1

This cell calculates token-level F1. Unlike Exact Match, F1 gives partial credit when the prediction overlaps with the reference answer.

In [ ]:
def token_f1(prediction, reference):

    pred_tokens = normalize_answer(prediction).split()
    ref_tokens = normalize_answer(reference).split()

    if not pred_tokens or not ref_tokens:
        return int(pred_tokens == ref_tokens)

    common = set(pred_tokens) & set(ref_tokens)

    if not common:
        return 0.0

    overlap = sum(
        min(pred_tokens.count(token), ref_tokens.count(token))
        for token in common
    )

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)

## 42. Run RAG evaluation

This cell evaluates the RAG model on the validation set and reports average Exact Match and F1 scores.

In [ ]:
em_scores = []
f1_scores = []

for example in evaluation_examples:

    question = example["question"]
    references = example["answers"][0]["text"]

    prediction, retrieved = generate_rag_answer(
        question,
        k=3
    )

    em_scores.append(
        exact_match(prediction, references)
    )

    f1_scores.append(
        max(
            token_f1(prediction, reference)
            for reference in references
        )
    )

print("RAG Exact Match:", sum(em_scores) / len(em_scores))
print("RAG F1:", sum(f1_scores) / len(f1_scores))

## 43. Inspect individual predictions

This cell displays individual questions, retrieved evidence, predictions, and reference answers so we can qualitatively inspect where retrieval helps or fails.

In [ ]:
for example in evaluation_examples[:10]:

    question = example["question"]
    references = example["answers"][0]["text"]

    prediction, retrieved = generate_rag_answer(
        question,
        k=3
    )

    print("\n" + "=" * 80)
    print("Question:", question)
    print("Retrieved:", retrieved[0]["title"])
    print("Prediction:", prediction)
    print("Reference:", references[0])

## 44. Compare retrieval quality with answer quality

This cell measures whether the answer's source passage appears in the retrieved Top-K documents. This separates retrieval errors from generation errors.

In [ ]:
retrieval_hits = 0

for example in evaluation_examples:

    question = example["question"]
    context = example["context"]

    retrieved = rag_retrieve(
        question,
        k=3
    )

    retrieved_contexts = [
        document["text"]
        for document in retrieved
    ]

    if context in retrieved_contexts:
        retrieval_hits += 1

retrieval_recall = (
    retrieval_hits / len(evaluation_examples)
)

print(
    f"Top-3 Retrieval Recall: "
    f"{retrieval_recall:.2%}"
)

## 45. Create a generator-only baseline

This cell defines a baseline that answers questions without retrieving any external document. The question is given directly to the same trained generator used by RAG.

In [ ]:
def generate_without_retrieval(query):

    inputs = tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        output_ids = generator.generate(
            **inputs,
            max_new_tokens=64
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

## 46. Compare one question

This cell compares the answer produced without retrieval against answers produced using retrieved documents.

In [ ]:
example = rag_val[0]

question = example["question"]

print("QUESTION:")
print(question)

print("\nWITHOUT RETRIEVAL:")
print(generate_without_retrieval(question))

for k in [1, 3, 5]:

    results = rag_generate(question, k=k)

    print(f"\nWITH TOP-{k} RETRIEVAL:")
    print(results[0]["answer"])

## 47. Evaluate the no-retrieval baseline

This cell evaluates the generator-only baseline using the same validation questions and the same EM/F1 metrics used for RAG.

In [ ]:
baseline_em = []
baseline_f1 = []

for example in evaluation_examples:

    question = example["question"]
    references = [
        example["answers"][0]["text"]
    ]

    prediction = generate_without_retrieval(
        question
    )

    baseline_em.append(
        exact_match(
            prediction,
            references
        )
    )

    baseline_f1.append(
        max(
            token_f1(prediction, reference)
            for reference in references
        )
    )

print(
    "Baseline Exact Match:",
    sum(baseline_em) / len(baseline_em)
)

print(
    "Baseline F1:",
    sum(baseline_f1) / len(baseline_f1)
)

## 48. Evaluate different Top-K values

This cell evaluates RAG with different numbers of retrieved documents. Comparing Top-1, Top-3, and Top-5 shows how the amount of retrieved evidence affects the model.

In [ ]:
rag_results = {}

for k in [1, 3, 5]:

    em_scores = []
    f1_scores = []

    for example in evaluation_examples:

        question = example["question"]
        references = [
            example["answers"][0]["text"]
        ]

        prediction, _ = generate_rag_answer(
            question,
            k=k
        )

        em_scores.append(
            exact_match(
                prediction,
                references
            )
        )

        f1_scores.append(
            max(
                token_f1(
                    prediction,
                    reference
                )
                for reference in references
            )
        )

    rag_results[k] = {
        "EM": sum(em_scores) / len(em_scores),
        "F1": sum(f1_scores) / len(f1_scores)
    }

    print(
        f"Top-{k} | "
        f"EM: {rag_results[k]['EM']:.4f} | "
        f"F1: {rag_results[k]['F1']:.4f}"
    )

## 49. Summarize the ablation results

This cell creates a compact comparison of the baseline and RAG variants. The results show how retrieval and the number of retrieved documents affect this miniature implementation.

In [ ]:
print("\n=== RAG ABLATION RESULTS ===")

print(
    f"No Retrieval | "
    f"EM: {sum(baseline_em) / len(baseline_em):.4f} | "
    f"F1: {sum(baseline_f1) / len(baseline_f1):.4f}"
)

for k, scores in rag_results.items():

    print(
        f"Top-{k} Retrieval | "
        f"EM: {scores['EM']:.4f} | "
        f"F1: {scores['F1']:.4f}"
    )

## 50. Inspect retrieval failures

This cell identifies validation examples where the correct source context was not retrieved in the Top-3 results. These cases help distinguish retrieval limitations from generator limitations.

In [ ]:
failures = []

for example in evaluation_examples:

    question = example["question"]
    context = example["context"]

    retrieved = rag_retrieve(
        question,
        k=3
    )

    retrieved_contexts = [
        document["text"]
        for document in retrieved
    ]

    if context not in retrieved_contexts:

        failures.append({
            "question": question,
            "expected_context": context,
            "retrieved": retrieved
        })

print("Top-3 retrieval failures:", len(failures))

for failure in failures[:5]:

    print("\nQuestion:")
    print(failure["question"])

    print("\nExpected context:")
    print(failure["expected_context"][:300])

    print("\nRetrieved:")
    for document in failure["retrieved"]:
        print("-", document["title"])

## 51. Understand the RAG-Token objective

RAG-Token marginalizes over the retrieved documents independently at every output position. Therefore, each generated token can use a different retrieved document.

In [ ]:
def rag_token_probability(
    token_log_probs,
    retrieval_log_probs
):
    """
    token_log_probs:
        Log p(y_i | x, z, y_<i)
        Shape: [num_documents]

    retrieval_log_probs:
        Log p(z | x)
        Shape: [num_documents]
    """

    joint_log_probs = (
        retrieval_log_probs +
        token_log_probs
    )

    return torch.logsumexp(
        joint_log_probs,
        dim=0
    )

## 52. Compare RAG-Sequence and RAG-Token

This cell illustrates the difference between the two RAG formulations. RAG-Sequence combines document probabilities at the sequence level, while RAG-Token performs the marginalization separately for every generated token.

In [ ]:
print("RAG-Sequence:")
print("One document is used for the complete generated sequence.")

print("\nRAG-Token:")
print("The document can change at every generated token.")

## 53. Retrieve multiple documents

This cell retrieves multiple candidate documents for one question. These documents represent the latent document set over which RAG-Token marginalizes.

In [ ]:
query = "What molecule stores hereditary information?"

retrieved = rag_retrieve(
    query,
    k=3
)

for rank, document in enumerate(
    retrieved,
    start=1
):
    print(
        f"{rank}. {document['title']} "
        f"| score={document['score']:.4f}"
    )

## 54. Generate candidate answers from each document

This cell generates an answer conditioned on each retrieved document. These candidate sequences demonstrate the different evidence sources available to the RAG-Token formulation.

In [ ]:
for rank, document in enumerate(
    retrieved,
    start=1
):

    generator_input = create_generator_input(
        query,
        document
    )

    inputs = tokenizer(
        generator_input,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        output_ids = generator.generate(
            **inputs,
            max_new_tokens=32
        )

    answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    print(f"\nDocument {rank}: {document['title']}")
    print("Candidate:", answer)

## 55. Demonstrate token-level marginalization

This cell demonstrates the mathematical operation used by RAG-Token: for each output token, probabilities from all retrieved documents are combined before selecting the token.

In [ ]:
num_documents = 3

example_token_log_probs = torch.log(
    torch.tensor([0.70, 0.20, 0.10])
)

retrieval_log_probs = torch.log(
    torch.tensor([0.60, 0.30, 0.10])
)

marginal_log_prob = rag_token_probability(
    example_token_log_probs,
    retrieval_log_probs
)

print(
    "Marginal token probability:",
    marginal_log_prob.exp().item()
)

## 56. Final comparison

This cell summarizes the two RAG formulations implemented or demonstrated in this project.

In [ ]:
print("RAG-Sequence")
print("- One latent document for the complete sequence")
print("- Sequence-level marginalization")

print("\nRAG-Token")
print("- Latent document can change per token")
print("- Token-level marginalization")